This is a small personal tutorial for setting up labels and training a one class PERCH v2.0 classifier through OpenSoundscape for audio recordings.

It is not meant to be a full formal guide, it mostly adapts Opensoundscape original documentation for my use case, maybe it comes handy to someone else stuck getting PERCH to run.

This tutorial asumes that you already have a Folder with audio files and its annotations from Raven software, following Opensoundscape documentation.

For my particular operating system Kubuntu 24.04, one of the most difficult parts, was finding the proper combination of packages for PERCH v2.0 to run in my environment (in GPU mode, hardware: NVIDIA GeForce RTX 4050), this combination made it work for me 

- bioacoustics-model-zoo==0.12.1
- opensoundscape==0.12.1
- numpy==1.26.4 

I do not upload the full .yml, because I don't know if it's relevant for your OS, if somebody needs it, raise an issue and I will upload it

I cannot stress out more that once you get it running export your environment to .yml, I learned that lesson the hard way!!!



Load packages

In [ ]:
#Imports
from opensoundscape.annotations import BoxedAnnotations
import bioacoustics_model_zoo as bmz
perch = bmz.Perch2()

import sklearn
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

#Utils
from datetime import datetime
from glob import glob
from pathlib import Path
from matplotlib import pyplot as plt


Build training, validation and test sets from Raven tables

In [ ]:
#The following  is for creating a set of BoxedAnnotations
#In the same folder where I have the Raven selection tables I have the corresponding audio following this naming convention
#SiteID_Datetime  examples:
#E01_20241011_231500.Table.1.selections.txt
#E01_20241011_231500.WAV

dataset_path = Path("Path to your folder with audio files and selection boxes")

#Creating a list of audio files and one for the boxed annotations, my naming convention was such that the table selections name 
# is the same than recordings name except for the "Table.1.selections.txt,(raven's default naming convention)"

selections = glob(f"{dataset_path}/*.txt")

# I had some files without annotation boxes, so this works as a quick way to keep
# only the .wav files that have matching annotation tables.

audio_files = [
    f.replace("Annotation_Files", "Recordings").replace(
        ".Table.1.selections.txt", ".WAV"
    )
    for f in selections
]

# Create the OpenSoundscape BoxedAnnotations object from the Raven files.

all_annotations = BoxedAnnotations.from_raven_files(
    selections, annotation_column="Species", audio_files=audio_files,
)

all_annotations.df.head()

# Define clip length and overlap.
# For PERCH, clips need to be 5 seconds long.
# My labels were made on 10-second audio fragments, but your case might be longer, this clipping part makes sure to keep it to 5s
# The clip overlap and min_label_overlap, choosing at your own convention, 
# in my experience, this can affect the performance of the model, 
# I am leaving the parameter I choose for my frog call (for reference a typical call duration is 0.36s)

clip_duration = 5
clip_overlap = 0.1 
min_label_overlap = 0.05 

# Create the clip-level label DataFrame from the annotations.

labels_df = all_annotations.clip_labels(
    clip_duration=clip_duration,
    clip_overlap=clip_overlap,
    min_label_overlap=min_label_overlap   
)

labels_df.head()

# Defining my test set, here is hardcoded, but the sites were chosen at random,
# these sites will not be "seen" by the training or validation of the model
# In my case the site information is contained in the file name, and hence in the index of the BoxedAnnotations

mask = labels_df.reset_index()["file"].apply(lambda x: ("E8" in x) or ("E10" in x) or ("E17" in x)).values
test_set = labels_df[mask]

# All remaining files will be used for training and validation.

train_and_val_set = labels_df.drop(test_set.index)

# Save the training/validation and test tables as .csv to have a record of the split.

train_and_val_set.to_csv("yourpath_train_val.csv")
test_set.to_csv("yourpath_test.csv")

# Using sklearn to split my set into validation and training

labels_train, labels_val = sklearn.model_selection.train_test_split(train_and_val_set[["your_class"]])
label_class = ["your_class"]



['Training_Validation_set_barton/Train_validation_sets/E10_20241010_033000.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E10_20241009_094500.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E20_20241012_024500.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E07_20241008_201500.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E05_20241009_130000.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E14_20241012_134500.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E06_20241008_131500.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/00_Extra_20241012_203000.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E14_20241012_213000.Table.1.selections.txt', 'Training_Validation_set_barton/Train_validation_sets/E02_20241011_190000.Table.1.selections.txt', 'Tra

Train custom classifier (one class)

In [ ]:
# Training the model

# Define the classes for the custom classifier.

perch.change_classes(label_class)

perch.initialize_custom_classifier(hidden_layer_sizes=[],new_classes=label_class)

# Train the classifier in one step: this first creates embeddings and then fits
# the classification head.
# In my case, data augmentation helped a lot, so it is worth experimenting here.
# The number of workers and batch size will depend heavily on your machine.

perch.train(
    train_df=labels_train,
    validation_df=labels_val,
    n_augmentation_variants=10,
    embedding_batch_size=64,  
    embedding_num_workers=12, 
    steps=1500,
)

# Make predictions on the validation set using the trained classifier.
preds = perch.predict(labels_val, batch_size=64, num_workers=10)

# Plot the score distributions for positive and negative clips.

plt.hist(preds[labels_val == True], bins=20, alpha=0.5, label="positives")
plt.hist(preds[labels_val == False], bins=20, alpha=0.5, label="negatives")
plt.legend()

# Calculate ROC AUC on the validation set.
roc_auc_score(labels_val.values, preds, average=None)

Test the model performance on the held-out test set (for one class)

In [ ]:

probs_test = perch.predict(test_set, batch_size=64, num_workers=10)
labels_test = test_set["your_class"].values
preds_test = (probs_test > 0.5).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(labels_test, preds_test, average="binary")
roc_auc = roc_auc_score(labels_test, probs_test)
cm = confusion_matrix(labels_test, preds_test)

print(f"Test set metrics for your_class:")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"ROC AUC:   {roc_auc:.3f}")
print(f"Confusion matrix:\n{cm}")


Saving the model, yey!

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M")
# I found very useful the recommendation of saving the model with the parameters used for training, and the time stamp
# The following is a name example
filename = f"perch2_shallow_aug10_1000_ovelap0.1_label0.05_{timestamp}.model"
perch.save(filename)